In [0]:
import os, requests, pandas as pd
KEY = dbutils.secrets.get(scope='fred', key='api_key')
def fred(series_id, start='2018-01-01'):
    r = requests.get('https://api.stlouisfed.org/fred/series/observations',
    params={'series_id': series_id, 'api_key': KEY, 'file_type': 'json',
    'observation_start': start})
    r.raise_for_status()
    df = pd.DataFrame(r.json()['observations'])[['date', 'value']]
    df['series_id'] = series_id
    df['value'] = pd.to_numeric(df['value'], errors='coerce')
    return df

In [0]:
pip install requests pandas

In [0]:
# Preferred: read the key from a secret scope (the key never appears in the notebook)
FRED_KEY = dbutils.secrets.get(scope="fred", key="api_key")

# Fallback if you can't create a secret scope: use a widget (type the key into the box at the top
# of the notebook). Don't hardcode the key, and clear the widget before sharing or syncing to GitHub.
# dbutils.widgets.text("fred_api_key", "")
# FRED_KEY = dbutils.widgets.get("fred_api_key")

CATALOG = "whriv"      # change these
SCHEMA  = "economics"
TABLE   = f"{CATALOG}.{SCHEMA}.bronze_fred_observations"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")

In [0]:
import requests, pandas as pd, time

SERIES = [
    "CPIAUCSL",
    "CPILFESL",
    "MEDCPIM157SFRBCLE",
    "TRMMEANCPIM157SFRBCLE",
    "SNDR",
    "NDR12MCD",
]

def fred_get(endpoint, **params):
    r = requests.get(
        f"https://api.stlouisfed.org/fred/{endpoint}",
        params={**params, "api_key": FRED_KEY, "file_type": "json"},
        timeout=30,
    )
    data = r.json()
    if r.status_code != 200:
        raise Exception(f"{endpoint} {params.get('series_id')}: {data.get('error_message')}")
    return data

def pull_series(series_id, start="2018-01-01", **extra):
    obs = fred_get("series/observations", series_id=series_id,
                   observation_start=start, **extra)["observations"]
    pdf = pd.DataFrame(obs)[["date", "value"]]
    pdf["series_id"] = series_id
    return pdf

In [0]:
for s in SERIES:
    try:
        m = fred_get("series", series_id=s)["seriess"][0]
        print(f"{s:24} {m['frequency_short']}  {m['observation_start']} to {m['observation_end']}  {m['title'][:60]}")
    except Exception as e:
        print(f"{s:24} PROBLEM: {e}")

In [0]:
from pyspark.sql import functions as F

frames, failed = [], []
for s in SERIES:
    try:
        frames.append(pull_series(s))
    except Exception as e:
        failed.append((s, str(e)))
    time.sleep(0.5)

print("failed:", failed)

pdf_all = pd.concat(frames, ignore_index=True)
sdf = (spark.createDataFrame(pdf_all.astype(str))
            .withColumn("ingested_at", F.current_timestamp()))

# Make reruns safe: remove these series' old rows, then append the fresh pull
if spark.catalog.tableExists(TABLE):
    ids = ", ".join(f"'{s}'" for s in pdf_all["series_id"].unique())
    spark.sql(f"DELETE FROM {TABLE} WHERE series_id IN ({ids})")

sdf.write.mode("append").saveAsTable(TABLE)

In [0]:
%sql
SELECT series_id,
       COUNT(*)  AS row_count,
       MIN(date) AS first_date,
       MAX(date) AS last_date,
       SUM(CASE WHEN value = '.' THEN 1 ELSE 0 END) AS missing_values
FROM whriv.economics.bronze_fred_observations
GROUP BY series_id

In [0]:
%sql
SELECT * 
FROM whriv.economics.bronze_fred_observations